In [1]:
from data import X_train, X_test, y_train, y_test, DIM
import torch
import plotly.express as px
import numpy as np



In [2]:
a = X_train[0]
b = y_train[0]
print(a.shape)
torch.tensor(a.toarray(), dtype=torch.float)
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("MPS is available")
elif torch.cuda.mps.is_available():
    device = torch.device("cuda")
    print("cuda is available")
else:
    device = torch.device("cuda")
    print("cuda and mps not detected")
print(device)


(1, 2048)
MPS is available
mps


In [3]:
print(torch.cuda.is_available())

False


In [4]:
b

0

In [13]:
import torch
import torch.nn as nn

class TransformerModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.pre_classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(DIM, 256),
        )

        self.enc = nn.TransformerEncoderLayer(
            d_model=256,
            nhead=16,
            batch_first=True
        )

        self.transformer_enc = nn.TransformerEncoder(
            encoder_layer=self.enc,
            num_layers=2
        )

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.2),
            nn.Linear(256, 1),
        )

    def forward(self, x):
        # x should be shape: (batch_size, 20000)

        x = x.unsqueeze(1)
        # shape: (batch_size, 1, 20000)

        x = self.pre_classifier(x)

        x = self.transformer_enc(x)
        # shape: (batch_size, 1, 20000)

        x = x.mean(dim=1)
        # shape: (batch_size, 20000)

        x = self.classifier(x)
        # shape: (batch_size, 1)

        return x

model = TransformerModel().to(device)
a = torch.tensor(X_train[0].toarray(), dtype=torch.float).to(device)


In [14]:
a.shape

torch.Size([1, 2048])

In [15]:
model(a)

tensor([[0.2763]], device='mps:0', grad_fn=<LinearBackward0>)

In [16]:

from torch.utils.data import Dataset, DataLoader


class TfidfDataset(Dataset):
    def __init__(self, X_sparse, y):
        self.X = X_sparse
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        # Get one sparse row
        x = self.X[idx]

        # Convert only this row to dense
        x = torch.tensor(x.toarray().squeeze(0), dtype=torch.float32)

        y = torch.tensor(self.y[idx], dtype=torch.float32)

        return x, y

train_dataset = TfidfDataset(X_train, y_train)
test_dataset = TfidfDataset(X_test, y_test)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=True)


len(train_loader), len(test_loader)



(5000, 1250)

In [18]:
from torch.optim import Adam
from tqdm import tqdm
optimizer = Adam(model.parameters(), lr=3e-4)
loss_fn = nn.BCEWithLogitsLoss()
model = model.to(device)
num_epochs = 10


def train_epoch():
    total_loss = 0
    model.train()
    total_corr = 0
    total_samples = 0
    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch).squeeze(1)

        loss = loss_fn(logits, y_batch)

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        batch_correct = (preds == y_batch).sum().item()
        batch_total = y_batch.size(0)
        total_corr += batch_correct
        total_samples += batch_total

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader), total_corr / total_samples

for epoch in tqdm(range(num_epochs)):
    loss, acc = train_epoch()
    print(f"Epoch {epoch + 1}, loss: {loss:.4f}, acc:{acc:.4f}")


 20%|██        | 1/5 [00:49<03:17, 49.41s/it]

Epoch 1, loss: 0.3110, acc:0.8649


 40%|████      | 2/5 [01:36<02:23, 47.85s/it]

Epoch 2, loss: 0.3063, acc:0.8668


 60%|██████    | 3/5 [02:21<01:33, 46.60s/it]

Epoch 3, loss: 0.3061, acc:0.8666


 80%|████████  | 4/5 [03:04<00:45, 45.27s/it]

Epoch 4, loss: 0.3008, acc:0.8688


100%|██████████| 5/5 [03:45<00:00, 45.19s/it]

Epoch 5, loss: 0.2968, acc:0.8713


In [19]:
def test_epoch():
    total_loss = 0
    model.eval()
    total_corr = 0
    total_samples = 0
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        logits = model(X_batch).squeeze(1)

        loss = loss_fn(logits, y_batch)

        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).float()
        batch_correct = (preds == y_batch).sum().item()
        batch_total = y_batch.size(0)
        total_corr += batch_correct
        total_samples += batch_total

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader), total_corr / total_samples

l, acc = test_epoch()
print(f"Test loss: {l:.4f}, acc: {acc:.4f}")

Test loss: 0.0731, acc: 0.8760
